In [32]:
!pip install catboost lightgbm xgboost

In [33]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score

from catboost import CatBoostClassifier

In [34]:
train = pd.read_csv("/content/Train_dataset.csv")
test = pd.read_csv("/content/Test_dataset.csv")

print(train.head())
print(train.shape)
print(test.shape)

      SEQN  RIAGENDR  PAQ605  BMXBMI  LBXGLU  DIQ010  LBXGLT  LBXIN age_group
0  73564.0       2.0     2.0    35.7   110.0     2.0   150.0  14.91     Adult
1  73568.0       2.0     2.0    20.3    89.0     2.0    80.0   3.85     Adult
2  73576.0       1.0     2.0    23.2    89.0     2.0    68.0   6.14     Adult
3  73577.0       1.0     2.0    28.9   104.0     NaN    84.0  16.15     Adult
4  73580.0       2.0     1.0    35.9   103.0     2.0    81.0  10.92     Adult
(1966, 9)
(312, 8)


In [35]:
print(train.isnull().sum())

SEQN         12
RIAGENDR     18
PAQ605       13
BMXBMI       18
LBXGLU       13
DIQ010       18
LBXGLT       11
LBXIN         9
age_group    14
dtype: int64


In [56]:
train = train.dropna(subset=["age_group"]).reset_index(drop=True)

In [57]:
X = train.drop("age_group", axis=1)
y = train["age_group"]

In [58]:
train_cleaned = train.dropna(subset=["age_group"])
X = train_cleaned.drop("age_group", axis=1)
y = train_cleaned["age_group"]

In [59]:
imputer = SimpleImputer(strategy='median')

# Re-fit and transform on the cleaned X data to ensure no NaNs remain
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
test_imputed = pd.DataFrame(imputer.transform(test), columns=test.columns)

In [60]:
print(train["age_group"].unique())

['Adult' 'Senior']


In [61]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print('Split successful. Training set size:', X_train.shape)

Split successful. Training set size: (1561, 8)


In [62]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="F1",
    verbose=100
)

model.fit(X_train, y_train)

0:	learn: 0.3113772	total: 2.74ms	remaining: 1.37s
100:	learn: 0.4397590	total: 173ms	remaining: 683ms
200:	learn: 0.5793872	total: 380ms	remaining: 565ms
300:	learn: 0.7444169	total: 548ms	remaining: 362ms
400:	learn: 0.8459770	total: 723ms	remaining: 179ms
499:	learn: 0.8893805	total: 899ms	remaining: 0us


CatBoostClassifier(depth=6, eval_metric='F1', iterations=500, learning_rate=0.05, loss_function='Logloss', verbose=100)

In [63]:
pred = model.predict(X_valid)

# Using average='weighted' to handle string labels ('Adult', 'Senior')
print("F1 Score:", f1_score(y_valid, pred, average='weighted'))

F1 Score: 0.7783242441706644


In [64]:
test_preds = model.predict(test_imputed)

submission = pd.DataFrame({
    'SEQN': test['SEQN'],
    'age_group': test_preds.flatten()
})

submission.to_csv('submission.csv', index=False)
print('Submission file saved successfully!')
submission.head()

Submission file saved successfully!


,SEQN,age_group
0,77017.0,Adult
1,75580.0,Adult
2,73820.0,Adult
3,80489.0,Adult
4,82047.0,Adult


In [65]:
test_preds = model.predict(test_imputed)

In [66]:
# Predict on test data
test_preds = model.predict(test_imputed)

# Create submission with the original SEQN and the categorical predictions
submission = pd.DataFrame({
    "SEQN": test["SEQN"],
    "age_group": test_preds.flatten()
})

submission.to_csv("submission.csv", index=False)
print("Submission saved successfully!")
submission.head()

Submission saved successfully!


,SEQN,age_group
0,77017.0,Adult
1,75580.0,Adult
2,73820.0,Adult
3,80489.0,Adult
4,82047.0,Adult


In [67]:
print(train["age_group"].unique())

['Adult' 'Senior']


In [68]:
print(submission.head())

      SEQN age_group
0  77017.0     Adult
1  75580.0     Adult
2  73820.0     Adult
3  80489.0     Adult
4  82047.0     Adult


In [78]:
print(submission["age_group"].unique())

[0 1]


In [79]:
submission.head()

,SEQN,age_group
0,77017.0,0
1,75580.0,0
2,73820.0,0
3,80489.0,0
4,82047.0,0


In [80]:
print(submission["age_group"].value_counts())

age_group
0    289
1     23
Name: count, dtype: int64


In [81]:
from sklearn.metrics import f1_score

pred = model.predict(X_valid)
# Added average='weighted' to handle string labels instead of binary 0/1
print("Validation F1:", f1_score(y_valid, pred, average='weighted'))

Validation F1: 0.7783242441706644


In [82]:
# Predict
test_preds = model.predict(test_imputed)

# Convert labels to integers
label_map = {
    "Adult": 0,
    "Senior": 1
}

test_preds = pd.Series(test_preds.flatten()).map(label_map)

# Create submission
submission = pd.DataFrame({
    "SEQN": test["SEQN"],
    "age_group": test_preds.astype(int)
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print(submission["age_group"].unique())

      SEQN  age_group
0  77017.0          0
1  75580.0          0
2  73820.0          0
3  80489.0          0
4  82047.0          0
[0 1]


In [83]:
print(submission.head())

      SEQN  age_group
0  77017.0          0
1  75580.0          0
2  73820.0          0
3  80489.0          0
4  82047.0          0


In [84]:
print(submission["age_group"].unique())

[0 1]


In [87]:
submission.to_csv("submission.csv", index=False)

In [86]:
from google.colab import files

files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>